# Simple Baselines

Before training neural networks, I will measure two simple baselines:

1. a majority-class classifier;
2. a text-only TF-IDF and logistic regression classifier.

Only the validation set is used for comparison. The test set remains
untouched.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
)

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.baselines import (
    LABEL_ORDER,
    build_majority_baseline,
    build_text_baseline,
    evaluate_classifier,
)

In [ ]:
processed_directory = (
    project_root
    / "data"
    / "processed"
)

train_data = pd.read_csv(
    processed_directory
    / "train_chart_claim_dataset.csv"
)

validation_data = pd.read_csv(
    processed_directory
    / "validation_chart_claim_dataset.csv"
)

print("Training rows:", len(train_data))
print(
    "Validation rows:",
    len(validation_data),
)

In [ ]:
pd.DataFrame(
    {
        "train": (
            train_data["label"]
            .value_counts()
        ),
        "validation": (
            validation_data["label"]
            .value_counts()
        ),
    }
).reindex(LABEL_ORDER)

## Majority baseline

The majority baseline always predicts the most frequent label from the
training set.

In [ ]:
majority_model = (
    build_majority_baseline()
)

majority_model.fit(
    train_data[["claim_text"]],
    train_data["label"],
)

(
    majority_metrics,
    majority_report,
    majority_matrix,
) = evaluate_classifier(
    majority_model,
    validation_data[["claim_text"]],
    validation_data["label"],
)

majority_metrics

## TF-IDF and logistic regression

TF-IDF converts the claim text into numerical features. Logistic
regression then predicts one of the three classes.

In [ ]:
text_model = build_text_baseline(
    random_state=42,
)

text_model.fit(
    train_data["claim_text"],
    train_data["label"],
)

(
    text_metrics,
    text_report,
    text_matrix,
) = evaluate_classifier(
    text_model,
    validation_data["claim_text"],
    validation_data["label"],
)

text_metrics

In [ ]:
baseline_results = pd.DataFrame(
    [
        {
            "model": "Majority baseline",
            **majority_metrics,
        },
        {
            "model": (
                "TF-IDF + Logistic Regression"
            ),
            **text_metrics,
        },
    ]
)

baseline_results

In [ ]:
plot_data = baseline_results.set_index(
    "model"
)[["accuracy", "macro_f1"]]

plot_data.plot(
    kind="bar",
    figsize=(8, 4),
)

plt.title(
    "Simple Baseline Validation Results"
)
plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Text baseline confusion matrix

The confusion matrix shows which labels are confused by the text-only
model.

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=text_matrix.values,
    display_labels=LABEL_ORDER,
).plot(
    xticks_rotation=20,
)

plt.title(
    "TF-IDF Logistic Regression"
)
plt.tight_layout()
plt.show()

In [ ]:
text_report

## Save validation results

The files below record the current baseline results. The test set is
still not evaluated.

In [ ]:
reports_directory = (
    project_root / "reports"
)

reports_directory.mkdir(
    parents=True,
    exist_ok=True,
)

baseline_results.assign(
    evaluation_split="validation"
).to_csv(
    reports_directory
    / "baseline_validation_metrics.csv",
    index=False,
)

text_report.to_csv(
    reports_directory
    / (
        "text_logistic_regression_"
        "validation_report.csv"
    )
)

text_matrix.to_csv(
    reports_directory
    / (
        "text_logistic_regression_"
        "validation_confusion_matrix.csv"
    )
)

print("Baseline reports saved.")

## Result

The majority baseline reaches about 0.33 validation accuracy.

The text-only TF-IDF and logistic regression model reaches about 0.78
validation accuracy and clearly performs better than the majority
baseline.

This result does not prove that the system understands chart images.
The next neural text model will be compared against this text baseline,
and the computer vision model will be evaluated separately.